In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split

# ---- Fix for truncated images ----
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---- GPU ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================
# Dataset class
# ==========================================
class ShipDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'ImageId']
        label = self.df.loc[idx, 'has_ship']
        img_path = os.path.join(self.img_dir, img_id)

        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new("RGB", (256, 256), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)

# ==========================================
# Transforms
# ==========================================
train_tfms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# ==========================================
# Load dataset info
# ==========================================
csv_path = "/kaggle/input/airbus-ship-detection/train_ship_segmentations_v2.csv"
img_dir = "/kaggle/input/airbus-ship-detection/train_v2"

df = pd.read_csv(csv_path)
df['has_ship'] = df['EncodedPixels'].notnull().astype(int)

# ---- Balanced subset: 35k ships + 35k no-ships = 70k ----
pos = df[df['has_ship'] == 1].sample(35000, random_state=42)
neg = df[df['has_ship'] == 0].sample(35000, random_state=42)
df_balanced = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

# ---- Train/validation split (90/10) ----
train_df, val_df = train_test_split(df_balanced, test_size=0.1, stratify=df_balanced['has_ship'], random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")  # Expect ~63k train, ~7k val

train_dataset = ShipDataset(train_df, img_dir, train_tfms)
val_dataset = ShipDataset(val_df, img_dir, val_tfms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

# ==========================================
# Model
# ==========================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print("✅ Model loaded on", device)

# ==========================================
# Training setup
# ==========================================
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = (outputs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze()
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

# ==========================================
# Training loop (30 epochs)
# ==========================================
for epoch in range(30):
    print(f"\nEpoch {epoch+1}/30")
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)
    print(f"Train Loss {tr_loss:.4f} | Acc {tr_acc:.4f}")
    print(f"Val   Loss {val_loss:.4f} | Acc {val_acc:.4f}")

torch.save(model.state_dict(), "cnn_ship_70k_30epochs.pth")
print("\n✅ Training complete! Model saved as cnn_ship_70k_30epochs.pth")


Using device: cuda
Train: 63000, Val: 7000
✅ Model loaded on cuda

Epoch 1/30


Train Loss 0.5104 | Acc 0.7575
Val   Loss 0.4624 | Acc 0.7984

Epoch 2/30


Train Loss 0.4376 | Acc 0.8039
Val   Loss 0.4129 | Acc 0.8120

Epoch 3/30


Train Loss 0.3915 | Acc 0.8273
Val   Loss 0.3791 | Acc 0.8411

Epoch 4/30


Train Loss 0.3587 | Acc 0.8458
Val   Loss 0.3462 | Acc 0.8583

Epoch 5/30


...

Train Loss 0.1689 | Acc 0.9356
Val   Loss 0.1952 | Acc 0.9293

Epoch 27/30


Train Loss 0.1655 | Acc 0.9357
Val   Loss 0.1995 | Acc 0.9309

Epoch 28/30


Train Loss 0.1627 | Acc 0.9368
Val   Loss 0.1972 | Acc 0.9254

Epoch 29/30


Train Loss 0.1613 | Acc 0.9380
Val   Loss 0.1971 | Acc 0.9299

Epoch 30/30


Train Loss 0.1583 | Acc 0.9397
Val   Loss 0.2066 | Acc 0.9337

✅ Training complete! Model saved as cnn_ship_70k_30epochs.pth
